# Train the current Roboflow dataset in Google Colab

This notebook loads your current dataset, installs the required Ultralytics package, and trains a YOLOv8 segmentation model using the extracted dataset structure.

### Setup Instructions for Google Colab

1. **Upload the dataset ZIP** to your Google Drive (if using ZIP)
   - File: `capstone dataset.v2-dataset2.0.yolov8.zip`
   - Target folder: `/MyDrive/`
   
   OR if you have the extracted dataset locally, the notebook will use the local path on first run, then you can use the extracted version.

2. **Open this notebook in Colab**:
   - Go to [colab.research.google.com](https://colab.research.google.com)
   - Click "File" → "Open notebook" → "Upload" and select this `.ipynb` file
   - Or paste this URL after uploading: `https://colab.research.google.com/`

3. **Run cells in order** starting from the top.

This notebook auto-detects Colab vs local and adjusts paths accordingly.

In [11]:
# Install Ultralytics
!pip install -U ultralytics

import os
import sys
import zipfile
from pathlib import Path

local_dataset_dir = Path(r'C:\wamp64\www\school_gate\cv_training\datasets\capstone_dataset.v2')
local_zip_path = Path(r'C:\Users\acer\Downloads\capstone dataset.v2-dataset2.0.yolov8.zip')

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    zip_path = Path('/content/drive/MyDrive/capstone dataset.v2-dataset2.0.yolov8.zip')
    data_dir = Path('/content/capstone_dataset.v2')
else:
    print('Running outside Colab. Using local Windows paths.')
    data_dir = local_dataset_dir
    zip_path = local_zip_path

if data_dir.exists() and any(data_dir.iterdir()):
    print(f'Using existing extracted dataset at {data_dir}')
else:
    if zip_path.exists():
        os.makedirs(data_dir, exist_ok=True)
        print(f'Extracting {zip_path} to {data_dir}...')
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(data_dir)
        print('Extraction complete')
    else:
        raise FileNotFoundError(
            f'No extracted dataset found at {data_dir} and no ZIP found at {zip_path}.\n'
            'Place the dataset ZIP at the path above or pre-extract it to the dataset folder.'
        )

# List key files to verify extraction or existing dataset
for p in sorted(data_dir.glob('*')):
    print(p)

DATA_DIR = data_dir

Running outside Colab. Using local Windows paths.
Using existing extracted dataset at C:\wamp64\www\school_gate\cv_training\datasets\capstone_dataset.v2
C:\wamp64\www\school_gate\cv_training\datasets\capstone_dataset.v2\data.yaml
C:\wamp64\www\school_gate\cv_training\datasets\capstone_dataset.v2\README.dataset.txt
C:\wamp64\www\school_gate\cv_training\datasets\capstone_dataset.v2\README.roboflow.txt
C:\wamp64\www\school_gate\cv_training\datasets\capstone_dataset.v2\test
C:\wamp64\www\school_gate\cv_training\datasets\capstone_dataset.v2\train
C:\wamp64\www\school_gate\cv_training\datasets\capstone_dataset.v2\valid


In [12]:
# Verify dataset structure and prepare the YAML path
from pathlib import Path

if not DATA_DIR.exists():
    raise FileNotFoundError(f'Dataset directory not found: {DATA_DIR}')

expected_files = ['data.yaml', 'train', 'valid']
for name in expected_files:
    path = DATA_DIR / name
    print(f'{name}:', 'FOUND' if path.exists() else 'MISSING')

if not (DATA_DIR / 'data.yaml').exists():
    raise FileNotFoundError(f'Dataset YAML missing from {DATA_DIR}')

print('Dataset ready for training.')

data.yaml: FOUND
train: FOUND
valid: FOUND
Dataset ready for training.


In [13]:
# Convert any YOLO bbox-only labels into segmentation polygons
from pathlib import Path

label_dirs = [DATA_DIR / subset / 'labels' for subset in ['train', 'valid', 'test']]
converted = 0
skipped = 0
for labels_dir in label_dirs:
    if not labels_dir.exists():
        print('Skipping missing label dir:', labels_dir)
        continue
    for label_file in sorted(labels_dir.glob('*.txt')):
        lines = label_file.read_text().strip().splitlines()
        new_lines = []
        changed = False
        for line in lines:
            parts = line.split()
            if len(parts) == 5:
                cls, x, y, w, h = parts
                try:
                    x, y, w, h = map(float, (x, y, w, h))
                except ValueError:
                    new_lines.append(line)
                    continue
                x1 = max(0.0, x - w / 2)
                y1 = max(0.0, y - h / 2)
                x2 = min(1.0, x + w / 2)
                y2 = min(1.0, y + h / 2)
                new_line = f"{cls} {x1:.6f} {y1:.6f} {x2:.6f} {y1:.6f} {x2:.6f} {y2:.6f} {x1:.6f} {y2:.6f}"
                new_lines.append(new_line)
                changed = True
                converted += 1
            else:
                new_lines.append(line)
        if changed:
            label_file.write_text('\n'.join(new_lines) + '\n')
            skipped += 1
print(f'Converted {converted} bbox lines in {skipped} label files.')

Converted 0 bbox lines in 0 label files.


In [15]:
# Train using the segmentation model and the current dataset
import sys
import torch
from ultralytics import YOLO

colab = 'google.colab' in sys.modules
project_dir = '/content/runs/detect' if colab else 'cv_training/runs/detect'

model = YOLO('yolov8n-seg.pt')

data_yaml = str(DATA_DIR / 'data.yaml')
if not (DATA_DIR / 'data.yaml').exists():
    raise FileNotFoundError(f'data.yaml not found in dataset directory: {data_yaml}')

# Auto-detect device and adjust batch size
if torch.cuda.is_available():
    device = '0'
    batch_size = 16
    workers = 4
    print('GPU detected: batch_size=16, workers=4')
else:
    device = 'cpu'
    batch_size = 2
    workers = 0
    print('CPU detected: batch_size=2, workers=0 (slow)')

print('Using device:', device)
print('Dataset YAML:', data_yaml)
print('Project output:', project_dir)

results = model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=batch_size,
    device=device,
    workers=workers,
    patience=20,
    project=project_dir,
    name='capstone_seg',
    exist_ok=True,
    verbose=True
)

print('Training finished')
print('Results saved to:', project_dir)

CPU detected: batch_size=2, workers=0 (slow)
Using device: cpu
Dataset YAML: C:\wamp64\www\school_gate\cv_training\datasets\capstone_dataset.v2\data.yaml
Project output: cv_training/runs/detect
New https://pypi.org/project/ultralytics/8.4.47 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.45  Python-3.12.10 torch-2.10.0+cpu CPU (AMD Ryzen 7 5700U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\wamp64\www\school_gate\cv_training\datasets\capstone_dataset.v2\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=Fa

In [ ]:
# Download the trained model (for Colab users only)
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    from google.colab import files
    model_path = Path('/content/runs/detect/capstone_seg/weights/best.pt')
    if model_path.exists():
        print('Downloading best.pt from Colab...')
        files.download(str(model_path))
    else:
        print('Model not found yet. Check training output above.')
else:
    model_path = Path('cv_training/runs/detect/capstone_seg/weights/best.pt')
    print('Local model saved to:', model_path)
    if model_path.exists():
        print('Model file size:', model_path.stat().st_size / 1e6, 'MB')



Local model saved to: cv_training\runs\detect\capstone_seg\weights\best.pt


: 